# Emotion Risk Engine V4

## Manual interactivo para ejecutar el backend en Google Colab

Este notebook prepara el repositorio, comprueba sus dependencias, valida una conexión de solo lectura a Neon, carga el modelo que realmente usa la API y permite exponer FastAPI temporalmente mediante Cloudflare Quick Tunnel.

**Repositorio:** [AvariAny/emotion-risk-engine](https://github.com/AvariAny/emotion-risk-engine)  
**Frontend:** [avaris-lab.vercel.app](https://avaris-lab.vercel.app/)

> Ejecuta las celdas una por una. Este archivo no ejecutó ninguna celda ni confirma que el flujo haya funcionado en Colab. Quick Tunnel es temporal y está pensado para pruebas y desarrollo, no como alojamiento permanente.

## Tabla de contenidos

1. Preparar Colab; revisar Python, PyTorch y GPU.
2. Clonar o preparar el repositorio sin sobrescribir archivos.
3. Instalar y verificar dependencias.
4. Leer DATABASE_URL desde Colab Secrets.
5. Probar Neon con SELECT 1.
6. Revisar configuración y disponibilidad de BETO.
7. Ejecutar una predicción local con texto ficticio.
8. Iniciar FastAPI y comprobar /health.
9. Crear un Quick Tunnel y detectar su URL.
10. Probar CORS para Vercel.
11. Probar /predict y validar el JSON.
12. Diagnóstico, apagado y cierre de sesión.

## Arquitectura general

Cliente en avaris-lab.vercel.app → URL temporal HTTPS de Cloudflare Quick Tunnel → FastAPI en Colab, puerto 8000 → BETO y Neon PostgreSQL.

La ruta /predict está declarada en app/api/routes.py y recibe JSON con el campo text. La respuesta incluye label, class, confidence y probabilities. /health devuelve status. El predictor de la API está en app/services/predictor.py y usa MODEL_DIR, DEVICE, MAX_LENGTH y LABELS desde app/utils/config.py.

Al importar app.api.main se ejecuta Base.metadata.create_all(bind=engine); puede crear las tablas declaradas que falten. /predict pasa una sesión SQLAlchemy al predictor y registra cada texto enviado en la tabla predictions. Por eso, las pruebas de API de este manual usan únicamente texto ficticio. La comprobación independiente de Neon ejecuta solo SELECT 1.

## Requisitos previos

- Abrir este archivo en Google Colab y seleccionar un runtime Python 3.10 o posterior. Una GPU ayuda, pero no es obligatoria.
- En el panel 🔑 Secrets de Colab guardar la cadena de conexión con el nombre DATABASE_URL y habilitar el acceso del notebook.
- La URL de Neon debe conservar la configuración TLS/SSL requerida por la base.
- Se necesita Internet para GitHub, paquetes, metadatos de Hugging Face y Cloudflare.
- Si el modelo no está en caché, la primera inferencia puede descargar sus pesos. El paso 6 consulta archivos locales o metadatos y no descarga los pesos.

### Dependencias según el repositorio

README recomienda requirements.txt. Ese archivo incluye FastAPI, Uvicorn, SQLAlchemy, psycopg2-binary, Pydantic y utilidades, pero no torch ni transformers. pyproject.toml sí los declara junto con dependencias de análisis y entrenamiento. Este manual instala requirements.txt y agrega torch o transformers solo si faltan, ya que son necesarios para el predictor de la API. No instala sentence-transformers ni paquetes de entrenamiento que el endpoint no necesita.

## 1. Preparar el runtime de Colab

**Qué haremos:** Mostrar Python, PyTorch, CUDA y GPU disponibles.

**Para qué sirve:** Conocer recursos antes de instalar o cargar el modelo.

**Requisitos:** Google Colab con Python 3.10 o posterior. GPU opcional.

**Resultado esperado:** Se indica versión de Python, versión de PyTorch si existe y dispositivo disponible.

**Si falla:** Si falta GPU, la inferencia puede usar CPU y tardar más. Si Python es anterior a 3.10, cambia el runtime.

**Precauciones de seguridad:** La celda solo lee versiones y disponibilidad; no consulta secretos ni envía datos.

In [ ]:
import platform
import sys

print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
if sys.version_info < (3, 10):
    raise RuntimeError("El proyecto requiere Python 3.10 o posterior.")

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA disponible:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("Se usará CPU; la inferencia puede tardar más.")
except ImportError:
    print("PyTorch aún no está instalado; se revisará en el paso de dependencias.")

Python: 3.13.16
Plataforma: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cpu
CUDA disponible: False
Se usará CPU; la inferencia puede tardar más.


## 2. Clonar o preparar el repositorio

**Qué haremos:** Clonar el repositorio oficial solo si la carpeta destino no existe; reutilizar un clon existente sin actualizarlo ni sobrescribirlo.

**Para qué sirve:** Tener el código y configuración del backend en Colab.

**Requisitos:** Acceso a GitHub y espacio temporal en /content.

**Resultado esperado:** El clon queda listo y el directorio de trabajo cambia a su raíz.

**Si falla:** Si la carpeta ya existe y no es un clon Git, la celda se detiene para evitar sobrescribirla. Revisa la carpeta o elige manualmente otra ruta.

**Precauciones de seguridad:** No se incluyen secretos. No se hace pull, checkout, reset, limpieza ni escritura sobre un clon existente.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/AvariAny/emotion-risk-engine.git"
REPO_DIR = Path("/content/emotion-risk-engine")

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise FileExistsError(
            f"{REPO_DIR} existe y no parece ser un clon Git. No se modificó."
        )
    origin_url = subprocess.run(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if origin_url.startswith("git@github.com:"):
        origin_url = "https://github.com/" + origin_url.removeprefix("git@github.com:")
    normalized_origin = origin_url.removesuffix(".git")
    expected_origin = REPO_URL.removesuffix(".git")
    if normalized_origin != expected_origin:
        raise RuntimeError("El clon existente no apunta al repositorio oficial esperado.")
    print("Se reutilizará el clon existente del repositorio oficial, sin actualizarlo.")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
    print("Clon inicial terminado.")

os.chdir(REPO_DIR)
print("Directorio de trabajo:", Path.cwd())

Clon inicial terminado.
Directorio de trabajo: /content/emotion-risk-engine


## 3. Instalar y verificar dependencias

**Qué haremos:** Instalar requirements.txt y agregar torch o transformers solo si faltan; luego importar módulos requeridos por la API.

**Para qué sirve:** El archivo requirements.txt no declara torch ni transformers aunque pyproject.toml sí los declara.

**Requisitos:** Repositorio preparado, conexión a Internet; la instalación puede tardar.

**Resultado esperado:** Las importaciones de FastAPI, Uvicorn, SQLAlchemy, psycopg2, torch y transformers terminan correctamente.

**Si falla:** Lee el primer error. Revisa Python y vuelve a ejecutar tras cualquier reinicio de runtime. No instales paquetes de entrenamiento para ejecutar la API.

**Precauciones de seguridad:** No se usan tokens ni secretos. Se evitan dependencias de entrenamiento innecesarias para este endpoint.

In [ ]:
import importlib
import importlib.metadata
import importlib.util
import subprocess
import sys

requirements_file = REPO_DIR / "requirements.txt"
if not requirements_file.is_file():
    raise FileNotFoundError("No se encontró requirements.txt.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
    check=True,
)
missing_ml_packages = [
    package
    for module_name, package in (("torch", "torch"), ("transformers", "transformers"))
    if importlib.util.find_spec(module_name) is None
]
if missing_ml_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", *missing_ml_packages],
        check=True,
    )

required_modules = ("fastapi", "uvicorn", "sqlalchemy", "psycopg2", "torch", "transformers")
for module_name in required_modules:
    module = importlib.import_module(module_name)
    distribution = "psycopg2-binary" if module_name == "psycopg2" else module_name
    try:
        version = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        version = getattr(module, "__version__", "versión no disponible")
    print(f"{module_name}: disponible ({version})")

fastapi: disponible (0.141.1)
uvicorn: disponible (0.52.4)
sqlalchemy: disponible (2.1.1)
psycopg2: disponible (2.9.13)
torch: disponible (2.11.0+cpu)
transformers: disponible (5.18.0)


## 4. Configurar DATABASE_URL con Colab Secrets

**Qué haremos:** Leer DATABASE_URL de Colab Secrets y asignarla al entorno del runtime para SQLAlchemy y FastAPI.

**Para qué sirve:** Evitar pegar credenciales en celdas, archivos del repositorio o salidas.

**Requisitos:** En 🔑 Secrets crea DATABASE_URL y habilita el acceso de este notebook.

**Resultado esperado:** La celda solo confirma que el secreto existe; nunca muestra ni escribe su valor en disco.

**Si falla:** Revisa el nombre exacto y los permisos del secreto en Colab. No pegues su valor en el código.

**Precauciones de seguridad:** El valor queda solo en memoria del proceso y en el entorno heredado por FastAPI durante esta sesión. No se imprime ni se crea un archivo .env.

In [ ]:
import os
from google.colab import userdata

try:
    database_url_value = userdata.get("DATABASE_URL")
except Exception:
    raise RuntimeError(
        "No se pudo leer DATABASE_URL desde Colab Secrets. "
        "Revisa que exista y que el notebook tenga permiso."
    ) from None

if not database_url_value:
    raise RuntimeError("Colab Secrets devolvió DATABASE_URL vacío.")

os.environ["DATABASE_URL"] = database_url_value
del database_url_value
print("DATABASE_URL está configurada en esta sesión; su valor no se mostró.")

DATABASE_URL está configurada en esta sesión; su valor no se mostró.


## 5. Probar Neon con SELECT 1

**Qué haremos:** Abrir una conexión SQLAlchemy y ejecutar solo SELECT 1.

**Para qué sirve:** Comprobar credenciales, red y TLS antes de iniciar FastAPI.

**Requisitos:** DATABASE_URL configurada y SQLAlchemy/psycopg2 instalados.

**Resultado esperado:** La consulta devuelve 1 y la conexión se cierra; no se leen tablas ni predicciones.

**Si falla:** Comprueba la URL vigente de Neon, el host/base, TLS/SSL y conectividad. No compartas la URL.

**Precauciones de seguridad:** La URL no se imprime. El error solo muestra el tipo de excepción y la consulta es de solo lectura.

In [ ]:
import os
from sqlalchemy.engine import make_url

url = make_url(os.environ["DATABASE_URL"])

if url.drivername == "postgresql":
    url = url.set(drivername="postgresql+psycopg2")
elif url.drivername == "postgres":
    url = url.set(drivername="postgresql+psycopg2")

os.environ["DATABASE_URL"] = url.render_as_string(hide_password=False)

print("Controlador configurado para esta sesión: psycopg2")

Controlador configurado para esta sesión: psycopg2


In [ ]:
from sqlalchemy import create_engine, text

neon_engine = create_engine(os.environ["DATABASE_URL"], pool_pre_ping=True)
try:
    with neon_engine.connect() as connection:
        select_result = connection.execute(text("SELECT 1")).scalar_one()
    if select_result != 1:
        raise RuntimeError("SELECT 1 devolvió un valor inesperado.")
    print("Conexión a Neon verificada con SELECT 1.")
except Exception as error:
    error_type = type(error).__name__
    raise RuntimeError(
        f"No se pudo verificar Neon ({error_type}). "
        "Revisa secreto, SSL y conectividad sin compartir la URL."
    ) from None
finally:
    neon_engine.dispose()

Conexión a Neon verificada con SELECT 1.


## 6. Revisar configuración y disponibilidad de BETO

**Qué haremos:** Leer MODEL_DIR, DEVICE, MAX_LENGTH y LABELS; comprobar archivos locales o consultar solo metadatos remotos.

**Para qué sirve:** Saber qué modelo cargaría el predictor antes de bajar pesos.

**Requisitos:** Repositorio y dependencias preparados; Internet para metadatos.

**Resultado esperado:** Se confirma la configuración y se detectan pesos locales o acceso a metadatos de Hugging Face.

**Si falla:** Si el repositorio no es visible, revisa el MODEL_DIR o los permisos. No continúes con inferencia hasta resolverlo.

**Precauciones de seguridad:** La consulta no descarga pesos, no imprime tokens ni accede a Neon.

In [ ]:
from pathlib import Path
from app.utils.config import DEVICE, LABELS, MAX_LENGTH, MODEL_DIR

print("MODEL_DIR configurado:", MODEL_DIR)
print("Dispositivo:", DEVICE)
print("Longitud máxima:", MAX_LENGTH)
print("Etiquetas:", LABELS)

model_path = Path(MODEL_DIR)
model_available = False
if model_path.is_dir():
    has_config = (model_path / "config.json").is_file()
    has_weights = (
        (model_path / "model.safetensors").is_file()
        or (model_path / "pytorch_model.bin").is_file()
        or any(model_path.glob("*.safetensors"))
    )
    model_available = has_config and has_weights
    print("Carpeta local detectada. config.json:", has_config, "| pesos:", has_weights)
else:
    from huggingface_hub import model_info
    try:
        model_metadata = model_info(MODEL_DIR)
        model_available = True
        print("Metadatos accesibles para:", model_metadata.id)
        print("Los pesos aún no se han descargado.")
    except Exception as error:
        print("No se pudieron consultar metadatos:", type(error).__name__)
        print("No se descargaron pesos; revisa identificador o acceso.")

if model_available:
    print("Disponibilidad de configuración/metadatos comprobada.")
else:
    print("Disponibilidad del modelo pendiente de resolver.")
    raise RuntimeError("No continúes hasta hacer accesible el modelo configurado.")

MODEL_DIR configurado: avarixo/emotion-risk-beto
Dispositivo: cpu
Longitud máxima: 128
Etiquetas: {0: 'Sin riesgo', 1: 'Riesgo bajo', 2: 'Riesgo moderado', 3: 'Riesgo alto'}
Metadatos accesibles para: avarixo/emotion-risk-beto
Los pesos aún no se han descargado.
Disponibilidad de configuración/metadatos comprobada.


## 7. Predicción local con texto ficticio

**Qué haremos:** Crear EmotionPredictor del módulo app.services.predictor y llamar predict sin sesión de base de datos.

**Para qué sirve:** Verificar que tokenizer y modelo carguen y que la predicción tenga los campos esperados.

**Requisitos:** La sección 6 debe indicar que el modelo es accesible. La primera carga puede descargar pesos grandes.

**Resultado esperado:** Se devuelve label, class, confidence y probabilities. La llamada local no guarda texto en Neon.

**Si falla:** Revisa MODEL_DIR, conectividad, almacenamiento y acceso al modelo. No pegues tokens en las celdas.

**Precauciones de seguridad:** El texto es inventado y no se pasa db al predictor; no crea una fila.

In [ ]:
from app.services.predictor import EmotionPredictor

predictor = EmotionPredictor()
texto_ficticio_local = (
    "Hoy tuve una jornada tranquila y voy a descansar antes de continuar mis actividades."
)
resultado_local = predictor.predict(texto_ficticio_local)

required_prediction_keys = {"label", "class", "confidence", "probabilities"}
if not required_prediction_keys.issubset(resultado_local):
    raise AssertionError("La estructura del predictor no coincide con la API.")

print("Predicción local completada con texto ficticio.")
print({key: resultado_local[key] for key in ("label", "class", "confidence")})
print("Clases en probabilidades:", list(resultado_local["probabilities"].keys()))

Cargando modelo BETO desde avarixo/emotion-risk-beto en memoria...


config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/729k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  439MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo cargado exitosamente (cpu)
Predicción local completada con texto ficticio.
{'label': 1, 'class': 'Riesgo bajo', 'confidence': 0.9997}
Clases en probabilidades: ['Sin riesgo', 'Riesgo bajo', 'Riesgo moderado', 'Riesgo alto']


## 8. Iniciar FastAPI y comprobar /health

**Qué haremos:** Lanzar Uvicorn en el puerto 8000 y esperar GET /health con JSON status igual a ok.

**Para qué sirve:** Comprobar que la API local está lista antes del túnel.

**Requisitos:** SELECT 1 y carga local del modelo completadas. Importar app.api.main ejecuta Base.metadata.create_all y crea el predictor de la API.

**Resultado esperado:** La URL local responde HTTP 200 con {status: ok}.

**Si falla:** Si vence el tiempo, revisa dependencias, Neon y modelo. El proceso se inicia sin mostrar logs para evitar exponer información sensible; para diagnóstico, deténlo e inicia Uvicorn manualmente en una celda visible.

**Precauciones de seguridad:** La importación puede crear tablas declaradas que falten, no las elimina. /predict se probará después con texto ficticio.

In [ ]:
import subprocess
import sys
import time
import requests

API_BASE_URL = "http://127.0.0.1:8000"
api_process = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn", "app.api.main:app",
        "--host", "0.0.0.0", "--port", "8000",
    ],
    cwd=str(REPO_DIR),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

health_response = None
for _ in range(180):
    if api_process.poll() is not None:
        break
    try:
        candidate = requests.get(f"{API_BASE_URL}/health", timeout=3)
        if candidate.status_code == 200:
            health_response = candidate
            break
    except requests.RequestException:
        pass
    time.sleep(1)

if health_response is None:
    process_status = api_process.poll()
    if process_status is None:
        raise TimeoutError("FastAPI no respondió en 180 segundos.")
    raise RuntimeError(
        f"Uvicorn terminó durante el inicio (código {process_status})."
    )
health_json = health_response.json()
if health_json != {"status": "ok"}:
    raise AssertionError("GET /health no devolvió el JSON esperado.")
print("GET /health: HTTP", health_response.status_code, health_json)

GET /health: HTTP 200 {'status': 'ok'}


## 9A. Instalar y verificar cloudflared

**Qué haremos:** Descargar el binario oficial según arquitectura si aún no existe y mostrar su versión.

**Para qué sirve:** Quick Tunnel necesita cloudflared junto al servidor local.

**Requisitos:** Internet y arquitectura Linux x86_64 o ARM64.

**Resultado esperado:** cloudflared --version termina correctamente.

**Si falla:** Si falla, revisa la arquitectura y la descarga oficial de Cloudflare.

**Precauciones de seguridad:** El binario queda en /content, fuera del clon. No usa tokens ni credenciales.

In [ ]:
import platform
import shutil
import subprocess
import urllib.request
from pathlib import Path

cloudflared_executable = shutil.which("cloudflared")
if cloudflared_executable is None:
    arch_name = platform.machine().lower()
    asset_by_arch = {
        "x86_64": "cloudflared-linux-amd64",
        "amd64": "cloudflared-linux-amd64",
        "aarch64": "cloudflared-linux-arm64",
        "arm64": "cloudflared-linux-arm64",
    }
    asset_name = asset_by_arch.get(arch_name)
    if asset_name is None:
        raise RuntimeError(f"Arquitectura no contemplada: {arch_name}")
    cloudflared_path = Path("/content/cloudflared")
    if not cloudflared_path.exists():
        download_url = (
            "https://github.com/cloudflare/cloudflared/releases/latest/download/"
            + asset_name
        )
        urllib.request.urlretrieve(download_url, cloudflared_path)
        cloudflared_path.chmod(0o755)
    cloudflared_executable = str(cloudflared_path)

version_output = subprocess.run(
    [cloudflared_executable, "--version"],
    check=True, capture_output=True, text=True,
)
print(version_output.stdout.strip())

cloudflared version 2026.10.1 (built 2026-10-15-17:08 UTC)


## 9B. Crear Quick Tunnel y detectar URL

**Qué haremos:** Ejecutar cloudflared hacia el servidor local y leer su salida en memoria para detectar el subdominio aleatorio trycloudflare.com.

**Para qué sirve:** Dar acceso temporal al backend desde el frontend remoto.

**Requisitos:** /health funciona y cloudflared está instalado.

**Resultado esperado:** Se detecta una URL HTTPS temporal mientras el proceso sigue activo.

**Si falla:** Si no aparece, confirma servidor, red y que la configuración local de cloudflared no interfiera. La URL cambia en cada sesión.

**Precauciones de seguridad:** La URL permite acceso externo mientras corre el proceso. Úsala solo para esta prueba y no envíes datos personales.

In [ ]:
from collections import deque
import re
import threading

tunnel_lines = deque(maxlen=100)
tunnel_process = subprocess.Popen(
    [
        cloudflared_executable, "tunnel", "--url", API_BASE_URL,
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

def collect_tunnel_output():
    for line in tunnel_process.stdout:
        tunnel_lines.append(line.rstrip())

tunnel_reader = threading.Thread(target=collect_tunnel_output, daemon=True)
tunnel_reader.start()
tunnel_url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
API_PUBLIC_URL = None

for _ in range(90):
    match = tunnel_url_pattern.search("\n".join(tunnel_lines))
    if match:
        API_PUBLIC_URL = match.group(0)
        break
    if tunnel_process.poll() is not None:
        break
    time.sleep(1)

if API_PUBLIC_URL is None:
    raise RuntimeError("No se detectó la URL del túnel; revisa red y procesos.")
print("URL temporal de la API:", API_PUBLIC_URL)

URL temporal de la API: https://thorough-bedford-yours-proceedings.trycloudflare.com


## 10. Probar CORS para Vercel

**Qué haremos:** Enviar OPTIONS a /predict con origen https://avaris-lab.vercel.app y preflight POST con Content-Type.

**Para qué sirve:** Verificar cabeceras que requiere un navegador antes de enviar JSON.

**Requisitos:** La URL del túnel sigue activa.

**Resultado esperado:** HTTP 200 refleja Access-Control-Allow-Origin, permite POST y Content-Type.

**Si falla:** Revisa CORSMiddleware en app/api/main.py y reinicia FastAPI antes de repetir.

**Precauciones de seguridad:** La prueba no envía texto ni credenciales. requests no aplica CORS como navegador, por eso se verifican las cabeceras.

In [ ]:
VERCEL_ORIGIN = "https://avaris-lab.vercel.app"
preflight_response = requests.options(
    f"{API_PUBLIC_URL}/predict",
    headers={
        "Origin": VERCEL_ORIGIN,
        "Access-Control-Request-Method": "POST",
        "Access-Control-Request-Headers": "content-type",
    },
    timeout=30,
)
preflight_headers = {
    key.lower(): value for key, value in preflight_response.headers.items()
}
allowed_methods = {
    item.strip().upper()
    for item in preflight_headers.get("access-control-allow-methods", "").split(",")
}
allowed_headers = {
    item.strip().lower()
    for item in preflight_headers.get("access-control-allow-headers", "").split(",")
}
cors_ok = (
    preflight_response.status_code == 200
    and preflight_headers.get("access-control-allow-origin") == VERCEL_ORIGIN
    and "POST" in allowed_methods
    and "content-type" in allowed_headers
)
print("Preflight HTTP:", preflight_response.status_code)
print("Origen permitido:", preflight_headers.get("access-control-allow-origin"))
print("POST permitido:", "POST" in allowed_methods)
print("Content-Type permitido:", "content-type" in allowed_headers)
if not cors_ok:
    raise AssertionError("Preflight no satisface CORS para Vercel.")
print("Preflight CORS validado para el origen de producción.")

Preflight HTTP: 200
Origen permitido: https://avaris-lab.vercel.app
POST permitido: True
Content-Type permitido: True
Preflight CORS validado para el origen de producción.


## 11. Probar POST /predict y validar JSON

**Qué haremos:** Enviar JSON con un texto inventado por el túnel y validar HTTP, campos, rangos, etiquetas y probabilidades.

**Para qué sirve:** Probar la petición de extremo a extremo que consume el frontend.

**Requisitos:** CORS pasó y API, Neon y túnel siguen activos.

**Resultado esperado:** HTTP 200 y respuesta con label, class, confidence y probabilities coherentes con LABELS.

**Si falla:** Revisa proceso, modelo y conexión Neon. El código no imprime el cuerpo de error.

**Precauciones de seguridad:** El texto ficticio se guarda en predictions porque el endpoint persiste cada predicción. No envíes texto real ni consultes datos personales.

In [ ]:
import json
from app.utils.config import LABELS

fictional_api_text = (
    "Hoy me siento tranquilo y planeo salir a caminar con una amistad."
)
predict_response = requests.post(
    f"{API_PUBLIC_URL}/predict",
    json={"text": fictional_api_text},
    headers={"Origin": VERCEL_ORIGIN},
    timeout=180,
)
if predict_response.status_code != 200:
    raise RuntimeError(
        f"POST /predict devolvió HTTP {predict_response.status_code}; "
        "no se mostró el cuerpo de error."
    )
prediction_json = predict_response.json()
expected_keys = {"label", "class", "confidence", "probabilities"}
if not expected_keys.issubset(prediction_json):
    raise AssertionError("Faltan campos requeridos en el JSON.")
label_value = prediction_json["label"]
confidence_value = prediction_json["confidence"]
probability_values = prediction_json["probabilities"]

if not isinstance(label_value, int) or label_value not in LABELS:
    raise AssertionError("label no coincide con etiquetas configuradas.")
if prediction_json["class"] != LABELS[label_value]:
    raise AssertionError("class no coincide con LABELS.")
if not isinstance(confidence_value, (int, float)) or not 0 <= confidence_value <= 1:
    raise AssertionError("confidence debe estar entre 0 y 1.")
if set(probability_values) != set(LABELS.values()):
    raise AssertionError("probabilities no contiene las clases configuradas.")
if not all(0 <= value <= 1 for value in probability_values.values()):
    raise AssertionError("Hay probabilidades fuera del rango 0 a 1.")
if abs(sum(probability_values.values()) - 1.0) > 0.02:
    raise AssertionError("La suma de probabilidades se aleja de 1.")

print("POST /predict: HTTP", predict_response.status_code)
print("Respuesta validada:")
print(json.dumps(prediction_json, ensure_ascii=False, indent=2))

POST /predict: HTTP 200
Respuesta validada:
{
  "label": 1,
  "class": "Riesgo bajo",
  "confidence": 0.9997,
  "probabilities": {
    "Sin riesgo": 0.0002,
    "Riesgo bajo": 0.9997,
    "Riesgo moderado": 0.0001,
    "Riesgo alto": 0.0001
  }
}


## 12. Diagnóstico de errores frecuentes

| Síntoma | Qué revisar |
| --- | --- |
| Dependencias o Python | Confirma Python 3.10+; repite la instalación después de reiniciar runtime. |
| DATABASE_URL no disponible | Revisa el nombre en 🔑 Secrets y el permiso del notebook. Nunca pegues ni imprimas el valor. |
| Neon no conecta | Verifica URL vigente, TLS/SSL, host, base y conectividad. La prueba controlada es SELECT 1. |
| Modelo no accesible | Confirma MODEL_DIR y acceso a avarixo/emotion-risk-beto. Metadatos no descargan pesos. |
| FastAPI no responde | Revisa dependencias, create_all, modelo y Neon. Inicia Uvicorn en primer plano tras detener el proceso de esta guía; no compartas logs que puedan contener información sensible. |
| Túnel sin URL | Confirma FastAPI, red, cloudflared y que no haya una configuración local que interfiera. |
| Error CORS | Revisa origen exacto https://avaris-lab.vercel.app, respuesta OPTIONS y cabeceras allow-origin, allow-methods y allow-headers. CORS compara origen, no ruta. |
| POST falla | Revisa JSON con campo text, proceso, modelo y Neon. No compartas el cuerpo de error sin revisarlo. |

Para iniciar Uvicorn manualmente y ver errores, primero detén el proceso de la siguiente sección y ejecuta desde la carpeta del clon: python -m uvicorn app.api.main:app --host 0.0.0.0 --port 8000.

### Seguridad y datos

- No pegues DATABASE_URL, tokens ni contraseñas en celdas, comentarios o salidas.
- No consultes ni exportes predictions en este manual.
- Cada POST /predict exitoso persiste el texto y resultado; aquí se usan solo textos ficticios.
- El Quick Tunnel es público mientras está activo. No uses datos personales ni lo consideres alojamiento estable.

## 12. Apagar procesos y cerrar sesión

**Qué haremos:** Terminar cloudflared y Uvicorn, liberar memoria GPU y retirar DATABASE_URL del entorno.

**Para qué sirve:** Evitar que la API continúe expuesta al terminar.

**Requisitos:** Ejecutar tras las pruebas o cuando se abandone el flujo.

**Resultado esperado:** Ambos procesos quedan detenidos y DATABASE_URL se retira del entorno.

**Si falla:** Si un proceso excede el tiempo, se finaliza tras un intento ordenado. También se puede desconectar el runtime en el menú de Colab.

**Precauciones de seguridad:** La limpieza termina solo los procesos de esta guía y quita la variable del entorno; no borra archivos ni datos de Neon.

In [ ]:
def stop_process(process, process_name):
    if process is None or process.poll() is not None:
        print(f"{process_name}: ya estaba detenido.")
        return
    process.terminate()
    try:
        process.wait(timeout=10)
        print(f"{process_name}: detenido.")
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)
        print(f"{process_name}: finalizado tras exceder el tiempo.")

stop_process(tunnel_process, "Cloudflare Quick Tunnel")
stop_process(api_process, "Uvicorn")

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except ImportError:
    pass

os.environ.pop("DATABASE_URL", None)
print("DATABASE_URL retirada del entorno de esta sesión.")
print("Cuando termines, desconecta el runtime desde el menú de Colab.")

## Lista de comprobación final

Marca cada punto solo después de ejecutar y observar su comprobación:

- [ ] Python 3.10 o posterior; CPU/GPU identificada.
- [ ] Repositorio clonado o clon existente reutilizado sin sobrescribir.
- [ ] Dependencias instaladas e importaciones verificadas.
- [ ] DATABASE_URL obtenida desde Colab Secrets sin mostrar el valor.
- [ ] SELECT 1 devolvió 1; no se consultaron tablas ni predicciones.
- [ ] MODEL_DIR y disponibilidad revisados sin descargar pesos en la comprobación.
- [ ] Predictor ejecutado con texto ficticio sin sesión de base de datos.
- [ ] FastAPI respondió en /health con status ok.
- [ ] Quick Tunnel entregó una URL HTTPS temporal.
- [ ] Preflight permitió el origen de Vercel, POST y Content-Type.
- [ ] POST /predict devolvió HTTP 200 y pasó validación JSON.
- [ ] Se consideró que el POST persiste el texto ficticio en predictions.
- [ ] Uvicorn, cloudflared y DATABASE_URL se cerraron/retiraron.

**Estado de este archivo:** las pruebas en Colab están pendientes. No se ejecutaron las celdas al crear este notebook.

### Referencia

Cloudflare describe Quick Tunnels como URLs temporales para pruebas: [documentación oficial de Quick Tunnels](https://developers.cloudflare.com/cloudflare-one/networks/connectors/cloudflare-tunnel/do-more-with-tunnels/trycloudflare/).